<a href="https://colab.research.google.com/github/mie3adfalotaibi-cloud/thakaa-flow-training/blob/main/week2/day08/day08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Task 1

### 1. Garbage In, Garbage Out
If the data we give the model has mistakes, the results will also be wrong.

Example from my field (Thakaa Flow training): a system that calculates attendance points for trainees.
- The data: check-in times.
- The small mistake: a time written as 1000 instead of 10:00.
- The wrong decision: the model marks the trainee as late or gives wrong points, so the ranking is unfair.

### 2. Find the problems

| Problem | Where | Type | Decision | Reason |
|---|---|---|---|---|
| Missing age | Reem, age = NaN | Null | Fill with the median | Only one row, we don't want to lose it |
| Age 250 | Hind, age | Outlier | Check with the customer, then fix or delete | No human is 250 years old |
| Repeated order | Order 1002 twice | Duplicate | Delete one of them | Same data, the order is counted twice |
| riyadh and Riyadh | Noura, city | Inconsistency | Make all of them Riyadh | The computer sees them as two cities |
| "three hundred" | Lama, amount | Wrong type | Change it to the number 300 | We can't calculate with text |

### 3. Features and Target (a problem from Riyadh)
Problem: predict the waiting time in a clinic.
- Target: waiting time in minutes.
- Features:
  1. Appointment hour
  2. Day of the week
  3. Number of patients booked that day
  4. Number of doctors available
  5. Visit type (general or specialist)
- Column we won't use: patient name, because it doesn't affect waiting time.

### 4. Why does an AI engineer spend most of the time on data?
Real data always has missing values, duplicates and mistakes.
The model doesn't know the data is wrong, it learns from it as it is.
So if we don't clean it, even a strong model will give wrong results.

## Task 2

In [1]:
import numpy as np
import pandas as pd

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

NumPy: 2.4.6
Pandas: 2.2.3


### Prediction before running
1. `prices_list * 2` → [25, 18, 32, 25, 18, 32] (a list repeats itself)
2. `prices * 2` → [50 36 64] (an array multiplies every number)
3. `prices_list + [1, 1, 1]` → [25, 18, 32, 1, 1, 1] (lists are joined together)
4. `prices + np.array([1, 1, 1])` → [26 19 33] (arrays add number by number)

The difference: a list treats `*` and `+` as repeat and join. A NumPy array does real math on each element.

In [2]:
prices_list = [25, 18, 32]
prices = np.array(prices_list)

print(prices_list * 2)
print(prices * 2)
print(prices_list + [1, 1, 1])
print(prices + np.array([1, 1, 1]))

[25, 18, 32, 25, 18, 32]
[50 36 64]
[25, 18, 32, 1, 1, 1]
[26 19 33]


In [3]:
print("shape:", prices.shape)
print("ndim:", prices.ndim)
print("size:", prices.size)
print("dtype:", prices.dtype)

a = np.array([1, 2.5, 3])
b = np.array([1, "2", 3])
print(a, a.dtype)
print(b, b.dtype)

shape: (3,)
ndim: 1
size: 3
dtype: int64
[1.  2.5 3. ] float64
['1' '2' '3'] <U21


### Why does an array need one type?
In the first array, 1 and 3 became 1.0 and 3.0 (float), because of 2.5.
In the second array, all numbers became text ('1' '2' '3'), because of "2".
An array needs one type for all elements so NumPy can do fast math on all of them together.

In [5]:
!unzip -o archive.zip
!unzip -o "archive (1).zip"

Archive:  archive.zip
  inflating: insurance.csv           
Archive:  archive (1).zip
  inflating: test_Y3wMUE5_7gLdaTN.csv  
  inflating: train_u6lujuX_CVtuZ9i.csv  


##Task3

In [6]:
days = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
sales = np.array([1240, 980, 1105, 1320, 1875, 2410, 2150])

# 1) statistics
print("Total:", sales.sum())
print("Mean:", round(sales.mean(), 2))
print("Median:", np.median(sales))
print("Max:", sales.max(), "| Min:", sales.min())
print("Std:", round(sales.std(), 1))

# 2) best and worst day
print("Best day:", days[sales.argmax()], sales.max())
print("Worst day:", days[sales.argmin()], sales.min())

# 3) boolean mask: days above average
above = sales > sales.mean()
print("Above average:", above)
print("Days above average:", above.sum())
print("Their sales:", sales[above])

# 4) slicing: Friday and Saturday
weekend = sales[5:]
print("Weekend total:", weekend.sum(), "| share:", round(weekend.sum() / sales.sum() * 100, 1), "%")

# 5) VAT and next week target, without for
print("With VAT:", sales * 1.15)
print("Next week target:", sales * 1.10)

# 6) change from day to day
change = np.diff(sales)
print("Change:", change)
print("Biggest jump:", days[change.argmax() + 1], change.max())

Total: 11080
Mean: 1582.86
Median: 1320.0
Max: 2410 | Min: 980
Std: 516.8
Best day: Fri 2410
Worst day: Mon 980
Above average: [False False False False  True  True  True]
Days above average: 3
Their sales: [1875 2410 2150]
Weekend total: 4560 | share: 41.2 %
With VAT: [1426.   1127.   1270.75 1518.   2156.25 2771.5  2472.5 ]
Next week target: [1364.  1078.  1215.5 1452.  2062.5 2651.  2365. ]
Change: [-260  125  215  555  535 -260]
Biggest jump: Thu 555


### Decision: mean or median?
The mean (1582.86) is higher than the median (1320) because Friday and Saturday are very high and pull the mean up.
If the owner asks "how much do I sell on a normal day?", I would give the median (1320), because most days are close to it and it is not affected by the weekend.

## Task 4

In [7]:
branches = ["Riyadh", "Jeddah", "Dammam"]
days = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
sales = np.array([
    [1240,  980, 1105, 1320, 1875, 2410, 2150],
    [ 910,  870,  990, 1010, 1430, 1980, 2240],
    [ 640,  720,  680,  750,  990, 1210, 1080],
])

# 1) shape: 3 rows = 3 branches, 7 columns = 7 days
print("shape:", sales.shape)

# 2) one value, one row, one column
print("Jeddah on Thursday:", sales[1, 4])
print("Dammam full week:", sales[2])
print("All branches on Friday:", sales[:, 5])
print("All branches on Saturday:", sales[:, 6])

# 3) axis=1 sums each row (branch), axis=0 sums each column (day)
branch_total = sales.sum(axis=1)
day_total = sales.sum(axis=0)
print("Total per branch:", branch_total)
print("Total per day:", day_total)

# 4) best branch, best day, share of each branch
print("Best branch:", branches[branch_total.argmax()], branch_total.max())
print("Best day:", days[day_total.argmax()], day_total.max())
print("Share %:", np.round(branch_total / sales.sum() * 100, 1))

# 5) highest single sale: which branch and which day?
i = sales.argmax()
print("argmax index:", i)
print("Highest sale:", branches[i // 7], days[i % 7], sales.max())

# 6) Sunday sales per branch
print("Sunday:", sales[:, 0])

shape: (3, 7)
Jeddah on Thursday: 1430
Dammam full week: [ 640  720  680  750  990 1210 1080]
All branches on Friday: [2410 1980 1210]
All branches on Saturday: [2150 2240 1080]
Total per branch: [11080  9430  6070]
Total per day: [2790 2570 2775 3080 4295 5600 5470]
Best branch: Riyadh 11080
Best day: Fri 5600
Share %: [41.7 35.5 22.8]
argmax index: 5
Highest sale: Riyadh Fri 2410
Sunday: [1240  910  640]


### Comment: axis=0 vs axis=1
axis=1 adds across each row, so it gives the total of each branch.
axis=0 adds down each column, so it gives the total of each day.

### Decision: which branch to close on Sunday?
I suggest closing Dammam on Sunday.
Its Sunday sales are the lowest (640 SAR, vs 1240 Riyadh and 910 Jeddah),
and it is also the weakest branch for the whole week (6070 SAR, only 22.8% of total sales).
So closing it on Sunday saves costs and loses the least money.

## Task 5

In [8]:
depth = np.array([120, 340, 610, -1, 890, 1150, 1420, 99999, 1730, 2010,
                  2280, -1, 2560, 2840, 99999, 3120, 3390, 3650, -1, 3900])

# 1) mask for bad readings: -1 (signal lost) or 99999 (sensor error)
bad = (depth == -1) | (depth == 99999)
print("Bad readings:", bad.sum(), "| share:", bad.mean() * 100, "%")

# 2) mean with and without bad readings (~ flips the mask)
good = depth[~bad]
print("Mean with bad:", round(depth.mean(), 2))
print("Mean without bad:", round(good.mean(), 2))
print("The error multiplied the mean by:", round(depth.mean() / good.mean(), 1))

# 3) median and max depth (meters and feet)
print("Median:", np.median(good))
print("Max depth:", good.max(), "m =", round(good.max() * 3.28084, 1), "ft")

# 4) water pressure = depth / 10 + 1
pressure = good / 10 + 1
print("Pressure:", pressure)
print("Readings above 300:", (pressure > 300).sum())

# 5) descent speed per minute
speed = np.diff(good)
print("Change per reading:", speed)

Bad readings: 5 | share: 25.0 %
Mean with bad: 11500.25
Mean without bad: 2000.67
The error multiplied the mean by: 5.7
Median: 2010.0
Max depth: 3900 m = 12795.3 ft
Pressure: [ 13.  35.  62.  90. 116. 143. 174. 202. 229. 257. 285. 313. 340. 366.
 391.]
Readings above 300: 4
Change per reading: [220 270 280 260 270 310 280 270 280 280 280 270 260 250]


### Speed: was it constant?
The changes are between 220 and 310 meters, around 270 most of the time.
So the speed was almost constant (about 270 m per minute).
Note: when we remove bad readings, some steps cover two minutes, so a few differences may look bigger.

### Decision: delete, keep, or fill the bad readings?
I would not keep them, because 99999 is not a real depth and it destroys the mean.
I would not just delete them either, because the model needs a reading every minute to predict 10 minutes ahead.
I would fill them using the readings before and after (for example, the middle value between them),
because the submarine goes down at an almost constant speed, so the missing depth is easy to estimate.